# Build a RAG agent without the gateway

You will answer questions about the AcruxCore docs two ways — retrieve first and ask once, then
hand the retriever to the model as a tool and let it search on its own — with **every model call
going straight to OpenRouter**. Your provider key never reaches AcruxCore's servers.

RAG means *retrieval-augmented generation*: before asking the model, you look up the passages most
likely to answer the question and put them in the prompt.

What "without the gateway" costs and keeps is the real subject here. Step 1 covers it before any
code runs.

| Piece | What it does | Who runs it |
|---|---|---|
| `rag-chat` | prompt for the linear path: takes `context` and `question` | the platform stores it |
| `rag-chat-agent` | prompt for the agentic path: takes `question`, told to use a tool | the platform stores it |
| `search_docs` | embeds a query and searches the index | **your code** |
| the index | five docs pages, chunked, embedded, in Chroma | **your code**, in this process |
| the model call | OpenRouter, called directly by the SDK | **your process**, not ours |

Every cell runs against a real account, the real docs site and the real OpenRouter API. Nothing
here is faked or mocked.

**Two ways to do every step.** Each step that creates something has two headings:
**In the dashboard**, with the values to type, and **The same thing in code**, with a cell to run.
They are not two different features — the dashboard and the SDK call the same API, so the result is
identical. Pick either. Doing both is harmless, because every code cell looks for what already
exists before it creates anything.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Build a RAG agent without the gateway](https://docs.acruxcore.com/docs/tutorials/build-a-rag-agent-without-the-gateway)

---

## Step 0 — What you need before you start

**1. A personal API key.** **Account & keys → New key**. Copy it the moment it appears — that is
the only time the full value is shown.

**2. An OpenRouter API key.** Get one at [openrouter.ai](https://openrouter.ai/). It needs a few
cents of credit: this notebook embeds about forty text chunks and makes half a dozen chat calls.

**3. No gateway model.** This is the one notebook in the set that needs nothing registered under
**Gateway → Models**. The model ids here belong to OpenRouter, not to us.

**4. Four packages.** `chromadb` is a vector database that runs inside this process — no server to
start.

In [ ]:
%pip install -q --upgrade acruxcore chromadb requests beautifulsoup4

**Setup.** Set both keys and name everything this notebook will create.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your shell
before you start Jupyter, and treat this cell as a fallback.

In [1]:
import json
import os

# Better: export these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")
os.environ.setdefault("OPENROUTER_API_KEY", "sk-or-...")

#: OpenRouter model ids, not AcruxCore ones. Nothing looks these up on our side.
CHAT_MODEL = "google/gemini-3.7-flash"
EMBED_MODEL = "openai/text-embedding-3-small"

LINEAR_PROMPT = "rag-chat"
AGENT_PROMPT = "rag-chat-agent"

#: The five guides this notebook indexes. Any public URL would do.
DOC_URLS = {
    "prompts.md": "https://docs.acruxcore.com/docs/guides/version-a-prompt",
    "gateway.md": "https://docs.acruxcore.com/docs/guides/route-calls-through-the-gateway",
    "tracing.md": "https://docs.acruxcore.com/docs/guides/trace-an-llm-call",
    "tools.md": "https://docs.acruxcore.com/docs/guides/create-a-tool",
    "evaluation.md": "https://docs.acruxcore.com/docs/guides/evaluate-a-prompt",
}

QUESTION = "How do I register a new model on the gateway?"

# Do NOT print the base URL: the saved output would publish whatever host you ran against.

### Preflight

**Check.** Four things, in the order they fail. The fourth is worth reading even if it passes,
because the way OpenRouter reports embedding support is genuinely misleading.

In [2]:
import chromadb
import requests
from bs4 import BeautifulSoup

from acruxcore import AcruxCore, AcruxCoreError

hub = AcruxCore()          # reads ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL

OPENROUTER = "https://openrouter.ai/api/v1"
OPENROUTER_KEY = os.environ["OPENROUTER_API_KEY"]
AUTH = {"Authorization": f"Bearer {OPENROUTER_KEY}"}

# 1. Does the AcruxCore key work?
await hub.prompts.list(limit=1)
print("acruxcore key: ok")

# 2. Is the index able to run in this process?
print("chromadb:", chromadb.__version__, "| beautifulsoup4: ok")

# 3. Does the OpenRouter key work for chat?
chat_probe = requests.post(f"{OPENROUTER}/chat/completions", headers=AUTH, timeout=60,
                           json={"model": CHAT_MODEL, "max_tokens": 1,
                                 "messages": [{"role": "user", "content": "hi"}]})
print("openrouter chat:", "ok" if chat_probe.ok else f"FAILED {chat_probe.status_code}")

# 4. Does the OpenRouter key work for EMBEDDINGS? Ask the endpoint, not the model list.
embed_probe = requests.post(f"{OPENROUTER}/embeddings", headers=AUTH, timeout=60,
                            json={"model": EMBED_MODEL, "input": "probe"})
if embed_probe.ok:
    vector = embed_probe.json()["data"][0]["embedding"]
    print(f"openrouter embeddings: ok, {len(vector)} dimensions")
else:
    print(f"openrouter embeddings: FAILED {embed_probe.status_code}")

# The trap: embedding models are absent from OpenRouter's own model list, even though the
# endpoint serves them. Searching the list and finding nothing is not an answer.
listed = requests.get(f"{OPENROUTER}/models", timeout=60).json()["data"]
embed_in_list = [m["id"] for m in listed if "embedding" in m["id"]]
print(f"models advertised by GET /models: {len(listed)}; of those, embedding models: "
      f"{len(embed_in_list)}")

acruxcore key: ok
chromadb: 1.5.9 | beautifulsoup4: ok
openrouter chat: ok
openrouter embeddings: ok, 1536 dimensions
models advertised by GET /models: 421; of those, embedding models: 0


---

## Step 1 — What "without the gateway" actually means

### The general problem

Every LLM-ops platform has to decide where it sits. There are only two answers.

A **proxy** sits in the request path. Your app calls the platform, the platform holds your provider
key and calls the model, then hands the answer back. It sees every request, so it can price it, cap
it, cache it, and retry it against a different provider.

A **library** sits beside the request path. Your app calls the model itself and tells the platform
what happened afterwards. The platform never holds your key and never sees the traffic, so it can
record but not control.

Most tools force you to pick one for your whole application. That is the real problem — the two
shapes suit different calls in the same app.

### Where our case sits

One argument switches between them:

```python
PROVIDER = {"base_url": "https://openrouter.ai/api/v1", "api_key": os.environ["OPENROUTER_API_KEY"]}

await hub.gateway.chat(model, messages)                     # proxy: we call the model
await hub.gateway.chat(model, messages, provider=PROVIDER)   # library: the SDK calls OpenRouter
```

With `provider` set, the SDK POSTs to that `base_url` from your process. The `api_key` is sent to
OpenRouter as a bearer token and nowhere else. `hub` still points at AcruxCore for everything that
is not the model call.

### The direct answer: what you keep and what you lose

| | Gateway path | BYO path |
|---|---|---|
| Prompts, versions, aliases, `render()` | yes | **yes** |
| Tool catalog, `@acrux.tool`, auto-sync | yes | **yes** |
| Traces, spans, payloads | yes | **yes** |
| `prompt_version_id` lineage on the span | yes | **yes** |
| Dollar cost on the span | yes | **no** — we do not know your rate card |
| Budgets and rate limits | yes | no |
| Response caching | yes | no |
| Automatic fallback to another provider | yes | no |
| Server-side tool resolution | yes | no — schemas are inlined into the request |
| A `Default model` on the prompt version | yes | **no** — that field names a *gateway* model |

The last row is the one that surprises people, and Step 11 shows the error it causes.

### Three traps, all real

**1. `rendered.model` is `None` on this path**, because the version deliberately binds no model. So
every call needs `rendered.model or CHAT_MODEL`. Leave the fallback out and the SDK gets `None`.

**2. `costUsd` is always empty on a BYO span.** Tokens, latency, model and payloads are all
recorded. The price is not, because we never saw the rate card. Your provider's dashboard has the
bill.

**3. Nobody reports the trace but you.** On the gateway path the server writes the `llm` span even
if your process dies. Here the SDK holds spans in a background queue in *your* process, so a
notebook that never closes its client can lose the last few. Step 13 is not optional.

### The recommendation

Use BYO when the key must not leave your infrastructure, when you want to drop a network hop, or
when the endpoint is something we do not proxy — vLLM or Ollama on your own machine. Use the
gateway when you want spend control, caching or fallback routing. Both paths write the same traces,
so you can move one call at a time and keep your history.

---

## Step 2 — Create the linear prompt

`rag-chat` takes the retrieved passages as `context` and the question as `question`. Putting it on
the platform rather than in the script means you can reword it and ship without a redeploy.

### In the dashboard

**Prompts → New prompt**, then the **Editor** tab.

| Field | What to enter |
|---|---|
| **Name** | `rag-chat` |
| **Description** | `Answers a question from retrieved documentation passages.` |
| **Default model** | leave **empty** — see the note below |
| **System message** | the `LINEAR_SYSTEM` string in the next code cell |
| **User message** | `{{ question }}` |

![The rag-chat editor with the system message ending in a context placeholder and a user message that is just a question placeholder](https://docs.acruxcore.com/img/tutorials/build-a-rag-agent-without-the-gateway/02-rag-chat-prompt.png)

**Leave Default model empty.** That dropdown lists models from **Gateway → Models**, and this
notebook never touches the gateway. `google/gemini-3.7-flash` here is an OpenRouter id, so the script
passes it at call time instead.

### The same thing in code

**Setup.** Two separate checks on purpose. A prompt shell with zero versions is a real state, and
"the name exists" is not "it has content".

In [3]:
LINEAR_SYSTEM = (
    "You answer questions about AcruxCore using only the documentation excerpts below. "
    "If the excerpts do not contain the answer, say so plainly instead of guessing.\n\n"
    "Documentation:\n{{ context }}"
)


async def find_prompt_by_name(name: str):
    """The prompt with exactly this name, or None. A notebook helper, NOT an SDK function."""
    found = await hub.prompts.list(search=name, limit=100)
    return next((p for p in found.data if p.name == name), None)


async def create_prompt_if_missing(name: str, *, description: str, system: str) -> str:
    """Create the shell, then commit version 1 - but only where they are missing.

    A notebook helper, NOT an SDK function. It wraps the same two real calls,
    hub.prompts.create() and hub.prompts.commit_version(), and skips whichever already
    exists so this notebook can be re-run. Returns the prompt id.

    Note what it does NOT pass: model=. On the BYO path the version binds no model.
    """
    prompt = await find_prompt_by_name(name)
    if prompt is None:
        prompt = await hub.prompts.create(name=name, description=description)
        print(f"  + created shell {name}")
    else:
        print(f"  = shell {name} already exists")

    if (await hub.prompts.list_versions(prompt.id, limit=1)).total == 0:
        version = await hub.prompts.commit_version(
            prompt.id,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": "{{ question }}"}],
        )
        print(f"  + committed {name} v{version.version_number}, model={version.model!r}, "
              f"aliases={[a.alias for a in version.aliases]}")
    else:
        print(f"  = {name} already has a version")
    return prompt.id


linear_id = await create_prompt_if_missing(
    LINEAR_PROMPT,
    description="Answers a question from retrieved documentation passages.",
    system=LINEAR_SYSTEM,
)

  = shell rag-chat already exists
  = rag-chat already has a version


**Check.** What `render` gives back. Read the `model` line closely — that `None` is the whole
reason the calls below need a fallback.

In [4]:
sample = await hub.prompts.render(
    LINEAR_PROMPT, "production",
    {"context": "[tools.md]\nOpen the prompt and go to the Tools tab.",
     "question": "How do I attach a tool?"})

print("model:          ", sample.model)
print("version:        ", f"v{sample.version_number}")
print("version_id:     ", sample.version_id)
print("tools:          ", sample.tools)
print("system message: ", sample.messages[0]["content"][:120], "...")

model:           None
version:         v1
version_id:      6c561301-6815-4408-a9f7-82f2ed8ef8bf
tools:           []
system message:  You answer questions about AcruxCore using only the documentation excerpts below. If the excerpts do not contain the ans ...


---

## Step 3 — Create the agentic prompt

`rag-chat-agent` gets no `context` variable at all. Instead its system message tells the model to
look things up with a tool, as many times as it needs. That single difference is what separates the
two halves of this notebook.

### In the dashboard

**Prompts → New prompt**, same as Step 2.

| Field | What to enter |
|---|---|
| **Name** | `rag-chat-agent` |
| **Description** | `Answers questions about AcruxCore by searching the docs with a tool.` |
| **Default model** | leave **empty**, for the same reason |
| **System message** | the `AGENT_SYSTEM` string in the next code cell |
| **User message** | `{{ question }}` |

### The same thing in code

**Setup.** The same find-or-create helper, called once more.

In [5]:
AGENT_SYSTEM = (
    "You answer questions about AcruxCore. Use the search_docs tool to look things up "
    "before answering - search more than once if the first result is thin, or if the "
    "question has several parts. Answer only from what the tool returns, and say so "
    "plainly when it comes back without the answer."
)

agent_id = await create_prompt_if_missing(
    AGENT_PROMPT,
    description="Answers questions about AcruxCore by searching the docs with a tool.",
    system=AGENT_SYSTEM,
)

  = shell rag-chat-agent already exists
  = rag-chat-agent already has a version


---

## Step 4 — Point the SDK at OpenRouter

There is nothing to do in the dashboard for this step, and that is the point: BYO is a client-side
setting, invisible to us.

**Your app.** This is the entire feature. One dictionary, passed to every model call.

Any OpenAI-compatible endpoint works here — Groq, Together, a vLLM server, Ollama on your laptop.
Only `base_url` changes.

In [6]:
import acruxcore as acrux

PROVIDER: acrux.ProviderConfig = {
    "base_url": OPENROUTER,
    "api_key": OPENROUTER_KEY,
}

print("base_url:", PROVIDER["base_url"])
print("api_key: ", f"set, {len(PROVIDER['api_key'])} characters (never printed)")

base_url: https://openrouter.ai/api/v1
api_key:  set, 73 characters (never printed)


---

## Step 5 — Fetch and chunk the docs

**Your app.** Download each page, strip the navigation, and cut the text into overlapping windows.

The overlap matters. Cut at exactly 1000 characters and a sentence that straddles the boundary is
in neither chunk, so a question about it retrieves nothing useful. 150 characters of overlap means
every boundary sentence appears whole in at least one chunk.

In [7]:
def fetch_doc_text(url: str) -> str:
    """Download one docs page and return its readable body text."""
    res = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")
    container = soup.find("article") or soup.find("main") or soup.body
    for tag in container.find_all(["nav", "aside", "script", "style"]):
        tag.decompose()
    return container.get_text(separator="\n", strip=True)


def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 150) -> list:
    """Split text into overlapping character windows."""
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")
    chunks, step = [], chunk_size - overlap
    for start in range(0, len(text), step):
        window = text[start:start + chunk_size].strip()
        if window:
            chunks.append(window)
        if start + chunk_size >= len(text):
            break
    return chunks


DOC_IDS, DOCUMENTS, METADATAS = [], [], []
for source, url in DOC_URLS.items():
    body = fetch_doc_text(url)
    pieces = chunk_text(body)
    print(f"  fetched {source}: {len(body):,} chars -> {len(pieces)} chunks")
    for index, piece in enumerate(pieces):
        DOC_IDS.append(f"{source}_{index}")
        DOCUMENTS.append(piece)
        METADATAS.append({"source": source, "chunk": index})

print(f"\n{len(DOCUMENTS)} chunks from {len(DOC_URLS)} documents")

  fetched prompts.md: 7,988 chars -> 10 chunks
  fetched gateway.md: 5,358 chars -> 7 chunks
  fetched tracing.md: 4,845 chars -> 6 chunks
  fetched tools.md: 8,219 chars -> 10 chunks
  fetched evaluation.md: 8,595 chars -> 10 chunks

43 chunks from 5 documents


---

## Step 6 — Embed the chunks and load the index

**Your app.** Embeddings come from the same host and the same key as the chat model, so BYO needs
only one provider credential.

One line in here is not obvious. OpenRouter does not promise it returns embeddings in the order you
sent the inputs — each row carries an `index` instead. Zip the raw response against your chunk list
and a handful of chunks end up filed under the wrong text, which produces a retriever that is
subtly, silently wrong. Sort by the echoed `index` and the problem disappears.

In [8]:
def embed_texts(texts: list) -> list:
    """Embed a batch of strings through OpenRouter's OpenAI-compatible endpoint."""
    res = requests.post(f"{OPENROUTER}/embeddings", headers=AUTH, timeout=90,
                        json={"model": EMBED_MODEL, "input": texts})
    res.raise_for_status()
    rows = res.json()["data"]
    # OpenRouter does not promise input order, so sort by the echoed index.
    return [row["embedding"] for row in sorted(rows, key=lambda r: r["index"])]


EMBEDDINGS = []
for start in range(0, len(DOCUMENTS), 64):
    EMBEDDINGS.extend(embed_texts(DOCUMENTS[start:start + 64]))
print(f"embedded {len(EMBEDDINGS)} chunks, {len(EMBEDDINGS[0])} dimensions each")

# Chroma runs in this process. Nothing to install, nothing to start, nothing written to disk.
collection = chromadb.Client().get_or_create_collection(
    name="acrux_docs", metadata={"hnsw:space": "cosine"})
if collection.count() == 0:
    collection.add(ids=DOC_IDS, embeddings=EMBEDDINGS,
                   documents=DOCUMENTS, metadatas=METADATAS)
print("indexed:", collection.count(), "chunks")

embedded 43 chunks, 1536 dimensions each
indexed: 43 chunks


**Your app.** Retrieval is one embedding call and one vector search. The source name is prepended to
each passage so the model can say where an answer came from.

In [9]:
def retrieve_context(question: str, top_k: int = 4) -> str:
    """Find the chunks closest to a question and join them into one context block."""
    [query_vector] = embed_texts([question])
    found = collection.query(query_embeddings=[query_vector], n_results=top_k)
    return "\n\n".join(
        f"[{meta['source']}]\n{chunk}"
        for meta, chunk in zip(found["metadatas"][0], found["documents"][0]))


CONTEXT = retrieve_context(QUESTION)
print(f"question: {QUESTION}")
print(f"retrieved {len(CONTEXT):,} characters\n")
print(CONTEXT[:420], "...")

question: How do I register a new model on the gateway?
retrieved 4,053 characters

[gateway.md]
On this page
Route your app's LLM calls through the gateway
What you'll build:
one OpenAI-compatible endpoint that fronts your provider.
You'll register a credential and a model, verify it live in the Playground, then
call it from code. Switching providers later becomes a config change, not a code
change.
1. Add a provider credential
​
Go to
Gateway → Credentials → New credential
. AcruxCore is
BYOK
— yo ...


---

## Step 7 — Answer it linearly

**Your app.** Retrieve, then make exactly one model call.

The order here is deliberate. `hub.traces.ingest()` reports the retrieval as its own span and
returns the id of the trace it just opened; passing that id to `chat()` puts the model call in the
same trace. Do it the other way round and you get two traces that nobody can connect.

Why report the retrieval yourself? Because we never saw it. The SDK knows about the model call it
made for you. Whatever you did beforehand — a vector search, a SQL query, a cache lookup — is
yours to describe.

![The rag-linear trace: a Retrieval span named search_docs, then an LLM span for the model](https://docs.acruxcore.com/img/tutorials/build-a-rag-agent-without-the-gateway/03-linear-trace.png)

In [10]:
from datetime import datetime, timezone


def _now() -> str:
    return datetime.now(timezone.utc).isoformat()


async def ask_linear(question: str) -> tuple:
    """Retrieve, then call the model once. Returns (answer, trace_id)."""
    started = _now()
    context = retrieve_context(question)

    # Open the trace with the retrieval span, and keep the id it hands back.
    reported = await hub.traces.ingest({
        "name": "rag-linear",
        "spans": [{
            "spanId": "retrieval", "name": "search_docs", "kind": "retrieval",
            "status": "ok", "startTime": started, "endTime": _now(),
            "input": {"query": question}, "output": {"context": context},
        }],
    })

    rendered = await hub.prompts.render(
        LINEAR_PROMPT, "production", {"context": context, "question": question})

    result = await hub.gateway.chat(
        rendered.model or CHAT_MODEL,        # rendered.model is None on this path
        rendered.messages,
        provider=PROVIDER,                   # the SDK calls OpenRouter, not us
        prompt_version_id=rendered.version_id,
        trace={"trace_id": reported.trace_id},
    )
    return result.content or "", reported.trace_id


linear_answer, linear_trace = await ask_linear(QUESTION)
print("trace:", linear_trace)
print()
print(linear_answer[:700])

trace: 49c095f6-b434-4f49-bf7b-37e5f53bfa64

To register a new model on the gateway:

1. Go to **Gateway → Models → New model**.
2. Set the **Public name** (the identifier callers will use in the `model` field).
3. Select your provider **credential**.
4. Set the **Upstream model** (the upstream model ID, such as `openai/gpt-4o-mini`).


---

## Step 8 — Answer it agentically

The linear version always retrieves, exactly once, whether or not the question needs it. This
version hands the retriever to the model as a tool and lets it decide — including deciding to
search three times for a question with three parts.

**Your app.** `@acrux.tool` derives the tool's name, description and parameter schema from the
function's signature and docstring. `run_tool_loop` then calls the model, runs whatever tools it
asks for, feeds the results back, and repeats until the model stops asking.

Two BYO-specific things happen here that do not happen on the gateway path. There is no server to
resolve tool schemas, so the SDK inlines the full JSON Schema into every request. And the tool is
**synced into your catalog** on first use, which is why `search_docs` appears there afterwards with
its definition taken from this docstring.

`trace={"name": "rag-agentic"}` names the trace. `run_tool_loop` honours that key;
`hub.gateway.chat()` does not, which is why Step 7 got its name from `traces.ingest()` instead.

![The runToolLoop trace: an LLM span, a nested Tool span named search_docs, then a second LLM span](https://docs.acruxcore.com/img/tutorials/build-a-rag-agent-without-the-gateway/05-agentic-trace.png)

In [11]:
@acrux.tool
async def search_docs(query: str) -> str:
    """Search AcruxCore's documentation for relevant information.

    Args:
        query: The search query - a question or topic to look up.
    """
    return retrieve_context(query)


async def ask_agentic(question: str) -> tuple:
    """Let the model call search_docs as often as it wants. Returns (answer, trace_id, turns)."""
    rendered = await hub.prompts.render(AGENT_PROMPT, "production", {"question": question})
    result = await hub.gateway.run_tool_loop(
        rendered.model or CHAT_MODEL,
        rendered.messages,
        tools=[search_docs],
        provider=PROVIDER,
        prompt_version_id=rendered.version_id,
        trace={"name": "rag-agentic"},
    )
    return result.content, result.trace_id, result.iterations


agentic_answer, agentic_trace, turns = await ask_agentic(QUESTION)
print(f"trace: {agentic_trace}  ({turns} model calls)")
print()
print(agentic_answer[:700])

trace: 4c0be30a-6a81-4402-b161-00bfa8989690  (2 model calls)

To register a new model on the AcruxCore Gateway:

1. Navigate to **Gateway → Models → New model** in the dashboard.
2. Configure the following fields:
   * **Public name**: The identifier callers will use in requests (e.g., `support-model`).
   * **Credential**: Select the provider credential the model should use (configured under **Gateway → Credentials**).
   * **Upstream model**: The specific model ID expected by the upstream provider (e.g., `openai/gpt-4o-mini`).

Once registered, callers can reference the public name in requests to `POST /gateway/chat/completions`, allowing you to re-point or change upstream models without modifying client code. You can also test the newly registered m


**Your app.** A question with three separate parts, to show the loop doing what the linear path
cannot: searching more than once, on its own initiative.

In [12]:
MULTI = ("What is a prompt alias, how do I attach a tool to a prompt, "
         "and where do I see the tokens a call used?")

multi_answer, multi_trace, multi_turns = await ask_agentic(MULTI)
print(f"trace: {multi_trace}  ({multi_turns} model calls)")
print()
print(multi_answer[:700])

trace: 1240103a-ecf2-47c7-894a-8b140a75d0d3  (5 model calls)

Based on the AcruxCore documentation:

---

### 1. What is a prompt alias?
A **prompt alias** is a named pointer (such as `production` or `staging`) that targets a specific committed version of a prompt:
- When a prompt's first version (v1) is created, both `production` and `staging` aliases point to v1 by default.
- Committing a new version (e.g., v2) does **not** automatically update what the aliases point to—a commit never silently changes what is live.
- You can promote an alias to point to a new version (either in the dashboard or via the API/SDK using `promoteAlias` / `promote_alias`), allowing you to test new versions on `staging` without affecting `production`.

---

### 2. How to at


---

## Step 9 — Read the traces back

**Check.** Four things are worth confirming from the API rather than taking on trust, and one of
them is an absence.

Token counts and the number of model calls move on every run. The span *shape* is what is stable.

![The expanded LLM span showing the model, openrouter.ai as the provider, tokens and latency](https://docs.acruxcore.com/img/tutorials/build-a-rag-agent-without-the-gateway/04-llm-span-detail.png)

In [13]:
await hub.gateway.flush()          # drain the background span queue before reading


def every_span(spans):
    """Flatten the span tree. A tool span is a child of the model turn that asked for it."""
    for span in spans:
        yield span
        yield from every_span(span.children)


for label, trace_id in (("linear", linear_trace),
                        ("agentic", agentic_trace),
                        ("agentic, 3-part question", multi_trace)):
    detail = await hub.traces.get(trace_id)
    spans = list(every_span(detail.spans))
    print(f"{label}: {detail.trace.name}  spans={detail.trace.span_count}  "
          f"tokens={detail.trace.total_tokens}")
    for span in spans:
        print(f"    {span.kind:<10} {(span.model or span.name)[:34]:<34} "
              f"cost={span.cost_usd!r}  prompt_version={'yes' if span.prompt_version_id else 'no'}")

linear: rag-linear  spans=2  tokens=1771
    retrieval  search_docs                        cost=None  prompt_version=no
    llm        google/gemini-3.7-flash            cost=None  prompt_version=yes
agentic: rag-agentic  spans=3  tokens=1962
    llm        google/gemini-3.7-flash            cost=None  prompt_version=yes
    tool       search_docs                        cost=None  prompt_version=no
    llm        google/gemini-3.7-flash            cost=None  prompt_version=yes
agentic, 3-part question: rag-agentic  spans=9  tokens=14907
    llm        google/gemini-3.7-flash            cost=None  prompt_version=yes
    tool       search_docs                        cost=None  prompt_version=no
    llm        google/gemini-3.7-flash            cost=None  prompt_version=yes
    tool       search_docs                        cost=None  prompt_version=no
    llm        google/gemini-3.7-flash            cost=None  prompt_version=yes
    tool       search_docs                        cost=None

Read that output for three things.

**The retrieval span sits in the linear trace next to the model call**, because `ingest` opened the
trace and `chat` joined it. That is the pairing the gateway path gets for free and this path has to
be told about.

**`cost=None` on every `llm` span.** Not a bug and not a gap in the run — we price calls that went
through the gateway, where the rate card is known. Nothing went through the gateway here.

**`prompt_version=yes` on the model spans.** Passing `prompt_version_id` from `render()` links the
span back to the exact version that produced it, so **View traces for this prompt version** works
on a call we never saw. That is what makes "how did v3 do?" answerable after you ship v4.

**Check.** The tool also arrived in your catalog on its own, taken from the decorated function.

In [14]:
found = await hub.tools.list(search="search_docs", limit=50)
for tool in found.data:
    if tool.name != "search_docs":
        continue
    versions = await hub.tools.list_versions(tool.id, limit=10)
    print(f"{tool.name}: {versions.total} version(s)")
    for version in versions.data:
        # source is 'code' for a definition derived from a decorated function,
        # 'dashboard' for one somebody typed in the UI.
        print(f"  v{version.version_number}  source={version.source}  "
              f"{(version.description or '')[:64]}")

search_docs: 1 version(s)
  v1  source=code  Search AcruxCore's documentation for relevant information.


Tools are versioned like prompts, and the source of a version is recorded. A definition that came
from a decorated function is marked **Defined in code**; one someone typed in the dashboard is not.
If both exist, the dashboard version is still there to promote back to.

![The search_docs tool page marked Defined in code, with the version from the docstring above an earlier dashboard version](https://docs.acruxcore.com/img/tutorials/build-a-rag-agent-without-the-gateway/06-tool-catalog.png)

---

## Step 10 — Four ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code.

### Mistake 1 — forgetting `provider=`

**Broken on purpose.** The whole BYO switch is one keyword argument. Leave it out and the call goes
to our gateway, which looks `google/gemini-3.7-flash` up in *your* model registry and does not find it —
because that id belongs to OpenRouter and was never registered here.

The error is clear, but only if you know that a missing `provider` is what sends you down this
path.

In [15]:
rendered = await hub.prompts.render(
    LINEAR_PROMPT, "production", {"context": CONTEXT, "question": QUESTION})

try:
    await hub.gateway.chat(CHAT_MODEL, rendered.messages)   # broken on purpose: no provider=
except AcruxCoreError as err:
    print(f"{err.code}  HTTP {err.status_code}")
    print(json.dumps(err.body)[:220])

API_ERROR  HTTP 400
{"error": {"code": "MODEL_NOT_REGISTERED", "message": "Model 'google/gemini-3.7-flash' is not registered. Add it under Gateway \u2192 Models."}}


### Mistake 2 — passing `rendered.model` straight through

**Broken on purpose.** The version binds no model, on purpose, so `rendered.model` is `None`. Drop
the `or CHAT_MODEL` fallback and there is no model to send.

The SDK stops this before the request leaves, because a provider would not treat it as an error:
an absent model reads as licence to pick a default, and you would get a complaint about a model
you never named. The message names both ways to supply one.

In [16]:
print("rendered.model is:", rendered.model)

try:
    await hub.gateway.chat(rendered.model, rendered.messages,   # broken on purpose: no fallback
                           provider=PROVIDER)
except (AcruxCoreError, TypeError, AttributeError) as err:
    print(f"{type(err).__name__}: {getattr(err, 'code', '')} "
          f"HTTP {getattr(err, 'status_code', '-')}")
    print(str(err)[:200])

rendered.model is: None
AcruxCoreError: VALIDATION_ERROR HTTP None
acruxcore: a model is required. Pass one explicitly, or bind a default model to the prompt version so render() returns it — rendered.model is None otherwise.


### Mistake 3 — not threading the trace id

**Broken on purpose.** The quiet one. Report the retrieval, then call the model without passing the
trace id, and both halves work perfectly. They just land in two different traces, so nothing
connects the passages to the answer they produced.

Nothing errors here. The only symptom is a trace with one span in it.

In [17]:
started = _now()
context = retrieve_context("What is a prompt alias?")
orphan = await hub.traces.ingest({
    "name": "rag-linear-split",
    "spans": [{"spanId": "retrieval", "name": "search_docs", "kind": "retrieval",
               "status": "ok", "startTime": started, "endTime": _now(),
               "input": {"query": "What is a prompt alias?"}}],
})

rendered = await hub.prompts.render(
    LINEAR_PROMPT, "production",
    {"context": context, "question": "What is a prompt alias?"})
result = await hub.gateway.chat(rendered.model or CHAT_MODEL, rendered.messages,
                               provider=PROVIDER, max_tokens=60)
# Broken on purpose: no trace={"trace_id": orphan.trace_id}, so the llm span opens its own.
await hub.gateway.flush()

detail = await hub.traces.get(orphan.trace_id)
print(f"retrieval trace {detail.trace.name}: spans={detail.trace.span_count}")
print("the answer came back fine:", (result.content or "")[:80], "...")
print("...and its llm span is in some other trace, with no retrieval beside it.")

retrieval trace rag-linear-split: spans=1
the answer came back fine: `production ...
...and its llm span is in some other trace, with no retrieval beside it.


### Mistake 4 — a wrong provider key

**Broken on purpose.** The SDK reports a provider's refusal as `PROVIDER_ERROR`, with the HTTP
status the provider returned.

Note what this cell does **not** print: `err.body`. A provider's 401 body sometimes echoes the key
it received, so printing it into a notebook publishes a credential to anyone who reads the file.
Branch on `err.code` and log the status. Never log a failed auth body.

In [18]:
BAD_PROVIDER: acrux.ProviderConfig = {"base_url": OPENROUTER, "api_key": "sk-or-not-a-real-key"}

try:
    await hub.gateway.chat(CHAT_MODEL, [{"role": "user", "content": "hello"}],
                           provider=BAD_PROVIDER)      # broken on purpose: bad key
except AcruxCoreError as err:
    print(f"code={err.code}  status={err.status_code}")
    print("body deliberately not printed - it can contain the key that was sent")

code=PROVIDER_ERROR  status=401
body deliberately not printed - it can contain the key that was sent


---

## Step 11 — Close the client

**Your app.** This matters more on the BYO path than anywhere else. Spans live in a queue inside
*your* process until they are flushed, and no server is holding a copy. Exit without closing and
the last spans are simply gone.

In a script `async with AcruxCore() as hub:` handles it. A notebook has no block to leave, so do it
by hand.

In [19]:
await hub.gateway.aclose()
print("flushed")

flushed


---

## What you built

A retriever over five docs pages and two ways to use it — retrieve-then-ask, and let-the-model-
search — with every model call going straight to OpenRouter and every run still showing up as a
trace with prompt-version lineage on it.

### What of this actually ships

The provider dict, the retriever, and the two functions:

```python
import os, requests, chromadb
from datetime import datetime, timezone
import acruxcore as acrux
from acruxcore import AcruxCore

OPENROUTER = "https://openrouter.ai/api/v1"
AUTH = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"}
PROVIDER: acrux.ProviderConfig = {"base_url": OPENROUTER,
                                  "api_key": os.environ["OPENROUTER_API_KEY"]}
CHAT_MODEL, EMBED_MODEL = "google/gemini-3.7-flash", "openai/text-embedding-3-small"

collection = chromadb.Client().get_or_create_collection(
    name="acrux_docs", metadata={"hnsw:space": "cosine"})   # fill it at startup


def embed_texts(texts: list) -> list:
    res = requests.post(f"{OPENROUTER}/embeddings", headers=AUTH, timeout=90,
                        json={"model": EMBED_MODEL, "input": texts})
    res.raise_for_status()
    return [r["embedding"] for r in sorted(res.json()["data"], key=lambda r: r["index"])]


def retrieve_context(question: str, top_k: int = 4) -> str:
    [vector] = embed_texts([question])
    found = collection.query(query_embeddings=[vector], n_results=top_k)
    return "\n\n".join(f"[{m['source']}]\n{c}"
                        for m, c in zip(found["metadatas"][0], found["documents"][0]))


@acrux.tool
async def search_docs(query: str) -> str:
    """Search AcruxCore's documentation for relevant information.

    Args:
        query: The search query - a question or topic to look up.
    """
    return retrieve_context(query)


async def ask_linear(question: str) -> str:
    async with AcruxCore() as hub:
        started = datetime.now(timezone.utc).isoformat()
        context = retrieve_context(question)
        reported = await hub.traces.ingest({"name": "rag-linear", "spans": [{
            "spanId": "retrieval", "name": "search_docs", "kind": "retrieval",
            "status": "ok", "startTime": started,
            "endTime": datetime.now(timezone.utc).isoformat(),
            "input": {"query": question}, "output": {"context": context}}]})
        rendered = await hub.prompts.render("rag-chat", "production",
                                           {"context": context, "question": question})
        result = await hub.gateway.chat(rendered.model or CHAT_MODEL, rendered.messages,
                                        provider=PROVIDER,
                                        prompt_version_id=rendered.version_id,
                                        trace={"trace_id": reported.trace_id})
        return result.content or ""


async def ask_agentic(question: str) -> str:
    async with AcruxCore() as hub:
        rendered = await hub.prompts.render("rag-chat-agent", "production",
                                           {"question": question})
        result = await hub.gateway.run_tool_loop(
            rendered.model or CHAT_MODEL, rendered.messages, tools=[search_docs],
            provider=PROVIDER, prompt_version_id=rendered.version_id,
            trace={"name": "rag-agentic"})
        return result.content
```

Everything else was scaffolding:

- `find_prompt_by_name` and `create_prompt_if_missing` exist so this notebook can be re-run. They
  are notebook helpers, not SDK calls, and in a real project the dashboard does their job once.
- `fetch_doc_text` and `chunk_text` are an indexing job, not a request path. Run them on a
  schedule and keep the vectors somewhere persistent.
- every **Check** cell — the preflight, the `render` inspection, the trace walk, the catalog read —
  proves a step worked. None of it belongs in a request path.
- Step 10 is all deliberately broken, and it leaves two extra traces behind.

### The three lines that make this BYO

```python
provider=PROVIDER                    # the SDK calls OpenRouter from your process
rendered.model or CHAT_MODEL         # the version binds no gateway model
trace={"trace_id": reported.trace_id}   # you own the trace, so you thread it
```

Remove the first and you are back on the gateway path, with dollar cost and budgets and a model
registry to keep in step. Both are supported at the same time, per call.

### What this notebook left in your team

- two prompts at v1 with no model bound: `rag-chat` and `rag-chat-agent`
- one tool, `search_docs`, synced from the decorated function on its first call
- several traces: one linear, two agentic, plus the split ones Step 10 leaves behind

### Where to go next

- [Build a tool-calling agent in Python (SDK)](https://docs.acruxcore.com/docs/tutorials/build-a-tool-calling-agent-in-python-sdk)
  — the same `run_tool_loop`, on the gateway path, with several tools at once.
- [Route your app's LLM calls through the gateway](https://docs.acruxcore.com/docs/guides/route-calls-through-the-gateway)
  — the other side of this trade: one hop more, in exchange for budgets, caching and fallback.
- [Using sessions and traces](https://docs.acruxcore.com/docs/guides/using-sessions-and-traces)
  — group several runs into one session and search across them.